## Load silver table data

In [0]:
%pip install sentence-transformers scikit-learn
dbutils.library.restartPython()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import builtins

In [0]:
df = spark.table("virtue_foundation_dataset.silver.facilities_cleaned")

# Only keep valid rows
df = df.filter(col("audit_reason").isNull())

## GOLD TABLES

In [0]:
facilities_core = df.select(
    "unique_id",
    "name",
    "organization_type",
    "facilityTypeId",
    "operatorTypeId",
    "yearEstablished",
    "capacity",
    "numberDoctors",
    "description"
)

facilities_core.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("virtue_foundation_dataset.gold.facilities_core")

facilities_location = df.select(
    "unique_id",
    "address_line1",
    "address_line2",
    "address_city",
    "address_stateOrRegion",
    "address_country",
    "latitude",
    "longitude"
)

facilities_location.write.mode("overwrite").saveAsTable("virtue_foundation_dataset.gold.facilities")

contact = df \
    .withColumn("phone", explode_outer("phone_numbers")) \
    .withColumn("website", explode_outer("websites")) \
    .select("unique_id", "phone", "website", "email")

contact.write.mode("overwrite").saveAsTable("virtue_foundation_dataset.gold.facilities_contact")


## One hot encoding

In [0]:
array_cols = [
    "specialties", "procedure", "equipment", "capability"
]

# Optional: only handle nulls safely (no type changes)
for c in array_cols:
    df = df.withColumn(
        c,
        when(col(c).isNull(), array()).otherwise(col(c))
    )


from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

from sklearn.cluster import KMeans
import pandas as pd
import numpy as np


from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def find_best_k(embeddings, k_min=5, k_max=300, step=5):
    best_k = k_min
    best_score = -1

    for k in range(k_min, builtins.min(k_max, len(embeddings)), step):
        if k <= 1:
            continue
        
        kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
        labels = kmeans.fit_predict(embeddings)

        # silhouette needs >1 cluster and <= n-1 labels
        if len(set(labels)) > 1:
            score = silhouette_score(embeddings, labels)

            if score > best_score:
                best_score = score
                best_k = k

    return best_k

def ai_cluster_ohe(df, colname, table_prefix, k=None):
    
    exploded = df.select(
        "unique_id",
        explode_outer(col(colname)).alias("raw")
    ).dropna()

    distinct_vals = exploded.select("raw").distinct().toPandas()

    if len(distinct_vals) == 0:
        raise ValueError(f"No values found in {colname}")

    values = distinct_vals["raw"].astype(str).tolist()

    # embeddings
    embeddings = model.encode(values)

    # auto-pick k if not provided
    if k is None:
        k = find_best_k(embeddings, k_min=5, k_max=50, step=5)
        print(f"{colname}: auto-selected k = {k}")

    # clustering
    kmeans = KMeans(
        n_clusters=builtins.min(k, len(values)),
        random_state=42,
        n_init="auto"
    )

    clusters = kmeans.fit_predict(embeddings)
    distinct_vals["cluster"] = clusters

    mapping = spark.createDataFrame(distinct_vals)

    clustered = exploded.join(mapping, "raw", "left")

    clustered.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"virtue_foundation_dataset.gold.{table_prefix}")

    ohe = clustered.groupBy("unique_id") \
        .pivot("cluster") \
        .agg(lit(1)) \
        .na.fill(0)

    for c in ohe.columns:
        if c != "unique_id":
            ohe = ohe.withColumnRenamed(c, f"{table_prefix.upper()}_{c}")

    ohe.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"virtue_foundation_dataset.gold.{table_prefix}_ohe")

    return clustered, ohe

## OHE specific tables

In [0]:
equipment_df, equipment_ohe = ai_cluster_ohe(
    df, "equipment", "facilities_equipment"
)

specialties_df, specialties_ohe = ai_cluster_ohe(
    df, "specialties", "facilities_specialties"
)

procedures_df, procedures_ohe = ai_cluster_ohe(
    df, "procedure", "facilities_procedures"
)

capabilities_df, capabilities_ohe = ai_cluster_ohe(
    df, "capability", "facilities_capabilities"
)

## Core Table and Final Feature Table

In [0]:
facilities_core = df.select(
    "unique_id", "name", "organization_type",
    "facilityTypeId", "capacity", "numberDoctors", "description"
)

facilities_core.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("virtue_foundation_dataset.gold.facilities_core")

gold_features = facilities_core \
    .join(equipment_ohe, "unique_id", "left") \
    .join(specialties_ohe, "unique_id", "left") \
    .join(procedures_ohe, "unique_id", "left") \
    .join(capabilities_ohe, "unique_id", "left") \
    .na.fill(0)

gold_features.write.mode("overwrite").saveAsTable("virtue_foundation_dataset.gold.facilities_features")